In [ ]:
# Full feature code (feature-feature model )
# Thresholds based on arbitrary values for each feature 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import gzip
import warnings
warnings.filterwarnings('ignore')

print("Loading data for diagnostic modeling...")

#loading data
columns = [
    'karl_id', 'host_name', 'model_name', 'hardware_make', 'karl_last_seen',
    'auth_username', 'serial_number', 'group_id', 'tenant_id', 'platform',
    'metric_category', 'measure_name', 'time', 'p90_processor_time',
    'avg_processor_time', 'max_cpu_usage', 'p90_memory_utilization',
    'avg_memory_utilization', 'max_memory_usage', 'p10_battery_health',
    'avg_battery_health', 'cpu_count', 'memory_count', 'memory_size_gb',
    'driver_vendor', 'os', 'wifi_mac_add', 'driver_version', 'driver_date',
    'os_version', 'driver', 'agent_id', 'performance_status', 'device_status',
    'max_battery_temperature', 'avg_battery_temperature', 'p90_battery_temperature',
    'avg_cpu_temp', 'p90_cpu_temp', 'avg_battery_discharge', 'p90_battery_discharge',
    'avg_boot_time', 'p90_boot_time', 'uptime_days', 'total_app_crash'
]

chunk_size = 100000
sample_data = []

with gzip.open('000.gz', 'rt') as f:
    for i, chunk in enumerate(pd.read_csv(f, sep='|', names=columns, chunksize=chunk_size)):
        sample_data.append(chunk)
        if i >= 4:
            break

df = pd.concat(sample_data, ignore_index=True)

numeric_cols = [
    'avg_processor_time', 'max_cpu_usage', 'avg_memory_utilization',
    'max_memory_usage', 'avg_battery_health', 'cpu_count', 'memory_size_gb',
    'avg_cpu_temp', 'avg_boot_time', 'p90_boot_time', 'uptime_days',
    'total_app_crash'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} records")

# feature prep 
feature_cols = numeric_cols
model_df = df[feature_cols + ['host_name']].dropna()

#Creating diagnostic thresholds 

df['max_cpu_usage'] = (df['max_cpu_usage'] > 10) & (df['max_cpu_usage'] < 15)
    df['avg_memory_utilization'] = (df['avg_memory_utilization'] > 50) & (df['avg_memory_utilization'] < 65)
    df['cpu_count'] = (df['cpu_count'] > 6500) & (df['cpu_count'] < 7250)
    df['avg_cpu_temp'] = (df['avg_cpu_temp'] > 50) & (df['avg_cpu_temp'] < 75)
    df['memory_size_gb'] = (df['memory_size_gb'] >= 16) & (df['memory_size_gb'] <= 18)
    df['uptime_days'] = (df['uptime_days'] >= 26) & (df['uptime_days'] <= 28)

def diagnose_issue(row):
    if row['avg_cpu_temp'] > 50 and row['max_cpu_usage'] > 75:
        return "Thermal/CPU Issue"
    elif row['avg_memory_utilization'] > 0 or row['max_memory_usage'] > 95:
        return "Memory Pressure"
    elif row['avg_battery_health'] < 60:
        return "Battery Degradation"
    elif row['avg_boot_time'] > 60 or row['p90_boot_time'] > 90:
        return "Boot Performance Issue"
    elif row['uptime_days'] > 30:
        return "Needs Restart / Stability Issue"
    elif row['total_app_crash'] > 10:
        return "Application Instability"
    else:
        return "Visit ITSE"

model_df['issue'] = model_df.apply(diagnose_issue, axis=1)

print("\nIssue Distribution:")
print(model_df['issue'].value_counts())

# Splitting data 
X = model_df[feature_cols]
y = model_df['issue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Training random forest 

print("\nTraining Random Forest Diagnostic Model...")
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("\nRandom Forest Diagnostic Performance:")
print(classification_report(y_test, y_pred_rf))

#Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.title('Feature Importance for Diagnosis')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#Diagnose new computer (fake data)
print("\nDiagnosing a new device...")

new_device = pd.DataFrame([[
    20, 85, 92, 95, 75, 8, 16, 88, 25, 40, 5, 2
]], columns=feature_cols)

# Predict issue
prediction = rf_model.predict(new_device)[0]

# Predict probabilities
probs = rf_model.predict_proba(new_device)[0]

# Confidence score
confidence = np.max(probs) * 100

print("\nPredicted Issue:", prediction)
print(f"Confidence: {confidence:.1f}%")

#Surface top 3 possible issues
issue_probs = dict(zip(rf_model.classes_, probs))
sorted_issues = sorted(issue_probs.items(), key=lambda x: x[1], reverse=True)

print("\nTop Possible Issues:")
for issue, prob in sorted_issues[:3]:
    print(f"{issue}: {prob*100:.1f}%")